# 📊 Exploratory Data Analysis (EDA) — Fantasy Premier League 2024/2025

## 1. Introdução

Este notebook tem como objetivo realizar uma **Exploratory Data Analysis (EDA)** sobre um conjunto específico de dados provenientes do repositório público:

**Fantasy Premier League Data — by vaastav**  
(https://github.com/vaastav/Fantasy-Premier-League)

O objetivo desta EDA é compreender a estrutura e comportamento dos dados referentes à época **2024–2025**, garantindo uma visão clara sobre os jogadores, equipas, posições e evolução de performance ao longo das Gameweeks (GWs).  
A partir desta análise exploratória, serão retirados insights essenciais para fases posteriores do projeto, nomeadamente:

- Pré-processamento para modelação;
- Transformações necessárias para um **modelo de regras de associação**;
- Seleção de features relevantes;
- Identificação de padrões gerais da época.

---

## 2. Justificação da Escolha dos Datasets

O repositório contém dezenas de ficheiros para cada época, mas para os objetivos deste projeto selecionámos **apenas dois datasets fundamentais**, ambos presentes localmente na pasta `data/` do projeto:

### 📌 1. `cleaned_players.csv`
Contém **informação agregada por jogador** ao longo da época:
- Nome, equipa, posição;
- Estatísticas anuais consolidadas (pontos totais, minutos jogados, golos, assistências, etc.);
- Dados “globais” úteis para caracterizar jogadores.

**Porquê incluí-lo?**  
Porque permite entender perfis gerais dos jogadores e possibilita cruzar estatísticas globais com padrões semana-a-semana. Serve como base para segmentações, agregações e merges adicionais com as GWs.

### 📌 2. `merged_gws.csv`
Contém informação detalhada **jornada a jornada** (Gameweek-level) para cada jogador:
- Pontos por GW;
- Minutos jogados;
- Contribuições ofensivas/defensivas;
- Variáveis temporais importantes para padrões de associação;

**Porquê incluí-lo?**  
Porque qualquer análise de padrões necessita de granularidade temporal.  
Este dataset permite transformar cada "jogador–GW" numa transação para associação de regras.

---

## 3. Objetivos da EDA

Nesta EDA vamos:

### 🔎 Explorar a estrutura dos datasets:
- Número de linhas/colunas;
- Tipos de dados;
- Missing values;
- Variáveis chave.

### 👤 Analisar os jogadores:
- Distribuição por posição;
- Distribuição por clubes;
- Estatísticas descritivas (pontos, minutos, golos, etc.).

### 🕒 Analisar Gameweeks:
- Evolução semanal de performance;
- Consistência dos jogadores ao longo da época;
- Identificação de outliers.

### 🔗 Preparar dados para futuras fases:
- Criação de merges entre as tabelas;
- Identificação de features úteis para regras de associação.

---

## 4. Instalação e Importação das Bibliotecas

In [17]:
%pip install pandas matplotlib seaborn numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Bibliotecas carregadas com sucesso!")

Bibliotecas carregadas com sucesso!


---

## 5. Carregamento dos Datasets

Os ficheiros utilizados neste notebook encontram-se na pasta local:

```
data/cleaned_players.csv
data/merged_gws.csv
```

Vamos agora carregar os dados para DataFrames Pandas.

In [25]:
players_path = "data/cleaned_players.csv"
gws_path = "data/merged_gw.csv"

players_df = pd.read_csv(players_path)
gws_df = pd.read_csv(gws_path, on_bad_lines='skip')

print("Datasets carregados com sucesso!")
players_df.head(), gws_df.head()

Datasets carregados com sucesso!


(  first_name           second_name  goals_scored  assists  total_points  \
 0      Fábio       Ferreira Vieira             0        0             0   
 1    Gabriel     Fernando de Jesus             3        2            42   
 2    Gabriel  dos Santos Magalhães             3        2           117   
 3        Kai               Havertz             9        3            97   
 4       Karl                  Hein             0        0             0   
 
    minutes  goals_conceded  creativity  influence  threat  bonus  bps  \
 0        0               0         0.0        0.0     0.0      0    0   
 1      600               5       119.5      154.4   255.0      6  152   
 2     2363              22       208.8      584.6   287.0      9  459   
 3     1872              20       269.0      467.6   711.0     14  343   
 4        0               0         0.0        0.0     0.0      0    0   
 
    ict_index  clean_sheets  red_cards  yellow_cards  selected_by_percent  \
 0        0.0      

---

## 6. Exploração Inicial dos Dados

Antes de avançar para análises mais profundas, vamos:

- Ver o tamanho dos datasets
- Analisar tipos de dados
- Identificar valores em falta
- Inspecionar colunas disponíveis

In [26]:
print("Dimensões — Players:", players_df.shape)
print("Dimensões — Gameweeks:", gws_df.shape)

print("\nTipos de Dados — Players:")
print(players_df.dtypes)

print("\nTipos de Dados — Gameweeks:")
print(gws_df.dtypes)

print("\nValores em falta — Players:")
print(players_df.isna().sum())

print("\nValores em falta — Gameweeks:")
print(gws_df.isna().sum())

Dimensões — Players: (804, 19)
Dimensões — Gameweeks: (14178, 42)

Tipos de Dados — Players:
first_name              object
second_name             object
goals_scored             int64
assists                  int64
total_points             int64
minutes                  int64
goals_conceded           int64
creativity             float64
influence              float64
threat                 float64
bonus                    int64
bps                      int64
ict_index              float64
clean_sheets             int64
red_cards                int64
yellow_cards             int64
selected_by_percent    float64
now_cost                 int64
element_type            object
dtype: object

Tipos de Dados — Gameweeks:
name                           object
position                       object
team                           object
xP                            float64
assists                         int64
bonus                           int64
bps                             int64
clean_she

---